### <font color='blue'> PROJETO: AUTOMATIZAÇÃO DE RELATÓRIOS POR E-MAIL</font>

In [1]:
# Bibliotecas

import pandas as pd
import smtplib
import os
from email.message import EmailMessage
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font
from datetime import datetime

# --- CONFIGURAÇÕES ---
EMAIL_REMETENTE = "xxxxxxxxxxx@gmail.com"
SENHA_APP = "aaaa uuuu yyyy zzz" 
caminho_arquivo = "Vendas_Geral.xlsx"

# 1. CARREGAMENTO E TRATAMENTO
df_vendas = pd.read_excel(caminho_arquivo, sheet_name="Vendas")
df_contatos = pd.read_excel(caminho_arquivo, sheet_name="Responsaveis")

ordem_meses = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
               'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']

df_vendas['Data_Aux'] = pd.to_datetime(df_vendas['Data'])
meses_map = {i+1: mes for i, mes in enumerate(ordem_meses)}
df_vendas['Mes'] = df_vendas['Data_Aux'].dt.month.map(meses_map)
df_vendas['Mes'] = pd.Categorical(df_vendas['Mes'], categories=ordem_meses, ordered=True)
df_vendas['Data'] = df_vendas['Data_Aux'].dt.strftime('%d/%m/%Y')

dicionario_emails = dict(zip(df_contatos['UF'], df_contatos['Email']))
dicionario_nomes = dict(zip(df_contatos['UF'], df_contatos['Nome']))

# Gerar data e hora atual
agora = datetime.now()
timestamp_arquivo = agora.strftime("%d_%m_%Y_%Hh%M")
timestamp_assunto = agora.strftime("%d/%m/%Y %H:%M")

try:
    server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
    server.login(EMAIL_REMETENTE, SENHA_APP)

    for uf in df_vendas['UF'].unique():
        destinatario = dicionario_emails.get(uf)
        nome_gerente = dicionario_nomes.get(uf, "Gerente")
        
        if not destinatario:
            continue

        df_uf = df_vendas[df_vendas['UF'] == uf].copy()
        
        # --- GERAÇÃO DO ARQUIVO EXCEL ---
        arquivo_envio = f"Vendas_Geral_{uf}_{timestamp_arquivo}.xlsx"
        
        with pd.ExcelWriter(arquivo_envio, engine='openpyxl') as writer:
            df_uf.drop(columns=['Data_Aux', 'Mes']).to_excel(writer, sheet_name='Vendas', index=False)
            
            # Ajuste largura aba Vendas
            ws_vendas = writer.sheets['Vendas']
            for col in ws_vendas.columns:
                max_length = 0
                column = col[0].column_letter
                for cell in col:
                    if cell.value: max_length = max(max_length, len(str(cell.value)))
                ws_vendas.column_dimensions[column].width = max_length + 2

            # Tabelas dinâmicas
            df_vend_soma = df_uf.pivot_table(index='Vendedor', columns='Mes', values='Vendas', aggfunc='sum', fill_value=0, margins=True, margins_name='Total', observed=True).reset_index()
            df_vend_fat = df_uf.pivot_table(index='Vendedor', columns='Mes', values='Faturamento', aggfunc='sum', fill_value=0, margins=True, margins_name='Total', observed=True).reset_index()
            df_prod_soma = df_uf.pivot_table(index='Produto', columns='Mes', values='Vendas', aggfunc='sum', fill_value=0, margins=True, margins_name='Total', observed=True).reset_index()
            df_prod_fat = df_uf.pivot_table(index='Produto', columns='Mes', values='Faturamento', aggfunc='sum', fill_value=0, margins=True, margins_name='Total', observed=True).reset_index()

            # Posições na aba Resumo
            start_row_vendedor = 2
            col_vendas = 0
            col_faturamento = df_vend_soma.shape[1] + 2
            start_row_produto = start_row_vendedor + df_vend_soma.shape[0] + 4

            df_vend_soma.to_excel(writer, sheet_name='Resumo', index=False, startrow=start_row_vendedor, startcol=col_vendas)
            df_vend_fat.to_excel(writer, sheet_name='Resumo', index=False, startrow=start_row_vendedor, startcol=col_faturamento)
            df_prod_soma.to_excel(writer, sheet_name='Resumo', index=False, startrow=start_row_produto, startcol=col_vendas)
            df_prod_fat.to_excel(writer, sheet_name='Resumo', index=False, startrow=start_row_produto, startcol=col_faturamento)

            # Formatação aba Resumo
            ws_resumo = writer.sheets['Resumo']
            negrito = Font(bold=True, size=11)
            ws_resumo.cell(row=start_row_vendedor, column=col_vendas+1, value="VENDAS POR VENDEDOR").font = negrito
            ws_resumo.cell(row=start_row_vendedor, column=col_faturamento+1, value="FATURAMENTO POR VENDEDOR").font = negrito
            ws_resumo.cell(row=start_row_produto, column=col_vendas+1, value="VENDAS POR PRODUTO").font = negrito
            ws_resumo.cell(row=start_row_produto, column=col_faturamento+1, value="FATURAMENTO POR PRODUTO").font = negrito

            for col in ws_resumo.columns:
                max_length = 0
                column = col[0].column_letter
                for cell in col:
                    if cell.value: max_length = max(max_length, len(str(cell.value)))
                ws_resumo.column_dimensions[column].width = max_length + 2

        # --- CÁLCULO DE RESUMO PARA O CORPO DO E-MAIL ---
        fat_total = df_uf['Faturamento'].sum()
        qtd_total = df_uf['Vendas'].sum()
        
        # Produto mais vendido (Quantidade)
        prod_mais_vendido = df_uf.groupby('Produto')['Vendas'].sum().idxmax()
        
        # Melhor vendedor (Faturamento)
        vendedor_top_nome = df_uf.groupby('Vendedor')['Faturamento'].sum().idxmax()
        vendedor_top_fat = df_uf.groupby('Vendedor')['Faturamento'].sum().max()
        vendedor_top_qtd = df_uf.groupby('Vendedor')['Vendas'].sum().loc[vendedor_top_nome]

        # --- ENVIO DO E-MAIL ---
        msg = EmailMessage()
        msg['Subject'] = f"Relatório de Vendas - {uf} ({timestamp_assunto})"
        msg['From'] = EMAIL_REMETENTE
        msg['To'] = destinatario

        corpo = f"""Olá {nome_gerente},

Segue o relatório de vendas consolidado da UF {uf}.

📊 RESUMO DO PERÍODO:
• Faturamento Total: R$ {fat_total:,.2f}
• Total de Vendas (Qtd): {qtd_total}
• Produto mais vendido: {prod_mais_vendido}

🏆 DESTAQUE VENDEDOR:
• Vendedor(a): {vendedor_top_nome}
• Faturamento: R$ {vendedor_top_fat:,.2f}
• Vendas realizadas: {vendedor_top_qtd}

O arquivo detalhado segue em anexo.
Gerado em: {timestamp_assunto}."""

        msg.set_content(corpo)

        with open(arquivo_envio, 'rb') as f:
            msg.add_attachment(f.read(), maintype='application', 
                               subtype='vnd.openxmlformats-officedocument.spreadsheetml.sheet', 
                               filename=arquivo_envio)

        server.send_message(msg)
        print(f"✅ Enviado: {uf} - {arquivo_envio}")
        
        if os.path.exists(arquivo_envio):
            os.remove(arquivo_envio)

except Exception as e:
    print(f"❌ Erro: {e}")
finally:
    server.quit()

✅ Enviado: SP - Vendas_Geral_SP_06_05_2026_12h01.xlsx
✅ Enviado: RJ - Vendas_Geral_RJ_06_05_2026_12h01.xlsx
✅ Enviado: MG - Vendas_Geral_MG_06_05_2026_12h01.xlsx
